In [ ]:
import pandas as pd
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import recall_score
from datetime import datetime
from sklearn.linear_model import LogisticRegression

In [ ]:
train = pd.read_csv('train.csv')
test = pd.read_csv('test_x.csv')

In [ ]:
# train word embeddings using gensim library

from gensim.models import Word2Vec

train_docs = [train['text'].iloc[i].split(' ') for i in range(len(train['text']))] # gensim only takes list of word list as `sentences`
w2v_model = Word2Vec(sentences=train_docs, vector_size=100, window=5, negative=20, min_count=2, workers=1) # google its doc to tune the arguments
w2v_model.train(train_docs, total_examples=len(train_docs), epochs=10)

(520811, 795450)

In [ ]:
# check the learned word embeddings

# vocabulary
w2v_vocav = w2v_model.wv.index_to_key
print('example words in vocab: ', [w2v_vocav[i] for i in range(200)])

# example embedding
example_word = 'bomb'
word_vec = w2v_model.wv[example_word]
print('learned embedding of bomb: ', word_vec)

# BONUS POINT (0.5): show the top-10 words most similar to 'bomb'
# you may calculate manually, or refer to gensim function: https://radimrehurek.com/gensim/models/word2vec.html

most_similar = w2v_model.wv.most_similar(example_word, topn=10)
print(f'\nTop 10 words most similar to "{example_word}":')
for word, similarity in most_similar:
  print(f'{word}: {similarity:.4f}')

example words in vocab:  ['the', 'to', 'a', 'in', 'of', 'and', 'I', 'is', 'for', '-', '', 'on', 'you', 'The', 'my', 'that', 'with', 'by', 'at', 'it', 'from', 'be', 'was', 'are', 'have', 'this', 'like', '&amp;', 'A', 'as', 'just', 'up', 'your', 'but', 'me', 'out', 'so', "I'm", 'not', 'has', '??', 'after', 'will', 'via', 'an', 'get', 'or', 'about', 'when', 'into', 'over', 'all', '...', 'fire', '|', 'been', 'no', 'In', 'he', 'can', 'they', '2', 'people', 'we', 'if', 'who', 'i', 'than', 'do', 'more', 'one', "it's", 'his', 'what', "don't", 'how', 'To', 'This', 'new', 'her', 'now', 'would', 'got', 'killed', 'going', 'California', 'some', 'New', 'My', 'burning', 'off', 'had', '3', '????', 'there', 'video', 'Is', 'back', 'THE', 'see', 'You', 'buildings', 'know', '@YouTube', 'were', 'Full', 'More', 'If', 'still', 'time', 'go', 'day', 'think', 'love', "can't", 'police', 'down', 'say', 'their', 'RT', 'crash', 'suicide', 'car', 'body', 'How', 'Emergency', 'want', 'them', 'disaster', 'first', 'may'

In [ ]:
def preprocess_text(text):
    text = text.lower()  # Lowercase the text
    text = re.sub(r"http\S+", "", text)  # Remove URLs
    text = re.sub(r"[^a-zA-Z\s]", "", text)  # Remove punctuation and numbers
    text = re.sub(r"\s+", " ", text).strip()  # Remove extra whitespace
    return text

In [ ]:
train['clean_text'] = train['text'].apply(preprocess_text)
test['clean_text'] = test['text'].apply(preprocess_text)

In [ ]:
vectorizer = TfidfVectorizer(max_features=10000, ngram_range=(1,2))
X = vectorizer.fit_transform(train['clean_text'])
X_test = vectorizer.transform(test['clean_text'])
y = train['target']

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
model = LogisticRegression(max_iter=1000, C=0.7, class_weight='balanced')
model.fit(X_train, y_train)

LogisticRegression(C=0.7, class_weight='balanced', max_iter=1000)

In [ ]:
y_val_probs = model.predict_proba(X_val)[:, 1]  # Get P(disaster)
threshold = 0.4
y_val_pred = (y_val_probs >= threshold).astype(int)
recall = recall_score(y_val, y_val_pred)
print(f"Validation Recall at threshold {threshold}: {recall:.4f}")

Validation Recall at threshold 0.4: 0.8504


In [ ]:
test_preds = model.predict(X_test)

In [ ]:
submission = pd.DataFrame({
    'id': test['id'],
    'target': test_preds
})

In [ ]:
submission.to_csv('submission.csv', index=False)